# VELNOR Qwen3-8B QLoRA fine-tuning

This notebook performs adapter fine-tuning (not foundation-model pretraining) using 4-bit NF4 QLoRA. It targets free CUDA runtimes only. GPU availability and quotas are controlled by Colab; no paid compute is required or enabled by this notebook.

**Important:** Qwen3-8B is a large dense model. Even 4-bit QLoRA can run out of memory on a 16 GB T4. The preflight rejects GPUs below its memory thresholds, but passing preflight does not guarantee the run will complete. If Colab offers no free GPU, stop; do not enable paid compute.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select an available free GPU runtime; CPU training is disabled.')
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print('GPU:', p.name)
print('Total VRAM GiB:', round(p.total_memory / 1024**3, 2))
print('Free VRAM GiB:', round(free / 1024**3, 2))
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)
if p.total_memory < 14 * 1024**3 or free < 12 * 1024**3:
    raise RuntimeError('GPU does not meet Qwen3-8B QLoRA preflight. No training started.')

In [ ]:
import subprocess, os
REPO = '/content/velnor-qwen3-8b'
BRANCH = 'feat/qwen3-8b-qlora'
subprocess.run(['git','clone','--branch',BRANCH,'--depth','1','https://github.com/vectolith/velnor.git',REPO], check=True)
print('Checked out:', subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip())
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current'], text=True).strip())

In [ ]:
%cd /content/velnor-qwen3-8b
!python -m pip install -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available(), 'CUDA is not available after installation'
print('CUDA still available:', torch.cuda.get_device_name(0))

## Persist output to Google Drive

The run writes its adapter and provenance report directly to Drive. Training runs on the allocated cloud GPU, not on the iPhone.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess
drive.mount('/content/drive')
run_name = 'qwen3-8b-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'], text=True).strip()
RUN_DIR = Path('/content/drive/MyDrive/VELNOR-training-runs') / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(RUN_DIR / 'outputs')
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '256'
os.environ['VELNOR_QWEN3_EPOCHS'] = '3'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
print('Artifacts will be saved to:', RUN_DIR)

In [ ]:
%%bash
set -euo pipefail
cd /content/velnor-qwen3-8b/training
python -u train_qwen3_8b_qlora.py 2>&1 | tee "$VELNOR_QWEN3_OUTPUT/training.log"
test -s "$VELNOR_QWEN3_OUTPUT/adapter/adapter_config.json"
test -s "$VELNOR_QWEN3_OUTPUT/training_report.json"
echo 'Adapter and training report exist.'

In [ ]:
import hashlib, json
from pathlib import Path
root = Path(os.environ['VELNOR_QWEN3_OUTPUT'])
manifest = {}
for path in sorted(p for p in root.rglob('*') if p.is_file()):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    manifest[str(path.relative_to(root))] = h.hexdigest()
(root / 'SHA256SUMS.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('Run artifacts:', root)